<a href="https://colab.research.google.com/github/IKKNIGHT/MYC-PROTAC/blob/main/ProteinMPNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ── Cell 1: Mount Drive and load config ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import json, os, shutil, time
import numpy as np

DRIVE_DIR = '/content/drive/MyDrive/myc_gnn'

with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

print('Config loaded.')
print(f'Fixed motif:        {cfg["fixed_motif_str"]}')
print(f'Candidates to design: {len(cfg["rfdiff_top_candidates"])}')

Mounted at /content/drive
Config loaded.
Fixed motif:        A901,A927,A929,A947,A950
Candidates to design: 100


In [3]:
# ── Cell 2: Copy all inputs to local SSD ──────────────────────────────────────
LOCAL_DIR    = '/content/myc_local'
MPNN_IN_DIR  = '/content/mpnn_inputs'
MPNN_OUT_DIR = '/content/mpnn_outputs'
ESM_OUT_DIR  = '/content/esm_outputs'

for d in [LOCAL_DIR, MPNN_IN_DIR, MPNN_OUT_DIR, ESM_OUT_DIR]:
    os.makedirs(d, exist_ok=True)

# Copy config files
for fname in [
    'pipeline_config.json',
    'fixed_motif_residues.npy',
    'hotspot_residues_rfdiff.npy',
]:
    src = f'{DRIVE_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{LOCAL_DIR}/{fname}')
        print(f'  {fname}: OK')

# Copy all 100 candidate PDBs from Drive to local
MPNN_DRIVE = cfg['proteinmpnn_input_dir']
print(f'\nCopying candidate PDBs from Drive...')
start   = time.time()
copied  = 0
for fname in os.listdir(MPNN_DRIVE):
    if fname.endswith('.pdb'):
        shutil.copy2(f'{MPNN_DRIVE}/{fname}', f'{MPNN_IN_DIR}/{fname}')
        copied += 1

print(f'  {copied} PDB files copied in {time.time()-start:.1f}s')

# Reload config
with open(f'{LOCAL_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

print(f'\nReady. {copied} backbones queued for ProteinMPNN.')

  pipeline_config.json: OK
  fixed_motif_residues.npy: OK
  hotspot_residues_rfdiff.npy: OK

Copying candidate PDBs from Drive...
  100 PDB files copied in 1.1s

Ready. 100 backbones queued for ProteinMPNN.


In [4]:
# ── Cell 3: Install ProteinMPNN ────────────────────────────────────────────────
import subprocess, sys

def run(cmd, label=''):
    print(f'  {label or cmd[:60]}...', end=' ', flush=True)
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('FAILED')
        print(result.stderr[-1000:])
    else:
        print('OK')
    return result.returncode

import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU — switch to A100 before running ProteinMPNN')

# Clone ProteinMPNN
if not os.path.exists('/content/ProteinMPNN'):
    run('git clone https://github.com/dauparas/ProteinMPNN.git /content/ProteinMPNN',
        'Cloning ProteinMPNN')
else:
    print('  ProteinMPNN already cloned')

# Dependencies
run('pip install -q biopython',  'Installing Biopython')
run('pip install -q prody',      'Installing ProDy')

# Verify
try:
    import Bio
    print(f'  Biopython {Bio.__version__} — OK')
except:
    print('  Biopython — FAILED')

print('\nProteinMPNN ready.')

PyTorch 2.10.0+cu128 | CUDA 12.8
GPU: NVIDIA A100-SXM4-40GB
  Cloning ProteinMPNN... OK
  Installing Biopython... OK
  Installing ProDy... OK
  Biopython 1.86 — OK

ProteinMPNN ready.


In [5]:
# ── Cell 4: Parse chain layout from RFdiffusion outputs ───────────────────────
# RFdiffusion outputs three chains per design:
#   Chain A: original MYC residues 901-950 (fixed context)
#   Chain B: hallucinated scaffold (50-100 residues)
#   Chain C: hallucinated CPP tail (15 residues)
#
# ProteinMPNN will design sequences for chains B and C only.
# Chain A is fixed — these are the MYC-binding residues the GNN identified.

from Bio import PDB

parser      = PDB.PDBParser(QUIET=True)
sample_pdbs = sorted(os.listdir(MPNN_IN_DIR))[:3]

print('Chain layout verification:')
for fname in sample_pdbs:
    struct = parser.get_structure('x', f'{MPNN_IN_DIR}/{fname}')
    print(f'\n  {fname}:')
    for chain in struct[0]:
        res_list = list(chain.get_residues())
        print(f'    Chain {chain.id}: {len(res_list)} residues '
              f'({res_list[0].id[1]} — {res_list[-1].id[1]})')

# Store chain layout for ProteinMPNN commands
FIXED_CHAIN    = 'A'    # MYC context — do NOT design
SCAFFOLD_CHAIN = 'B'    # hallucinated scaffold — design with interface bias
CPP_CHAIN      = 'C'    # CPP tail — design with Arg/Leu bias
DESIGN_CHAINS  = f'{SCAFFOLD_CHAIN}{CPP_CHAIN}'

print(f'\nFixed chain (no design):  {FIXED_CHAIN} (MYC 901-950)')
print(f'Design chains:            {SCAFFOLD_CHAIN} (scaffold) + {CPP_CHAIN} (CPP tail)')

Chain layout verification:

  candidate_100_myc_binder_7.pdb:
    Chain B: 81 residues (1 — 81)
    Chain C: 15 residues (1 — 15)
    Chain A: 50 residues (901 — 950)

  candidate_10_myc_binder_31.pdb:
    Chain B: 67 residues (1 — 67)
    Chain C: 15 residues (1 — 15)
    Chain A: 50 residues (901 — 950)

  candidate_11_myc_binder_51.pdb:
    Chain B: 80 residues (1 — 80)
    Chain C: 15 residues (1 — 15)
    Chain A: 50 residues (901 — 950)

Fixed chain (no design):  A (MYC 901-950)
Design chains:            B (scaffold) + C (CPP tail)


In [52]:
# ── Cell 5: Build ProteinMPNN inputs — correct bias format ────────────────────
import glob, json, subprocess, os
import numpy as np

MPNN_SCRIPT   = '/content/ProteinMPNN/protein_mpnn_run.py'
HELPER_SCRIPT = '/content/ProteinMPNN/helper_scripts/parse_multiple_chains.py'
ASSIGN_SCRIPT = '/content/ProteinMPNN/helper_scripts/assign_fixed_chains.py'

NUM_SEQS    = 8
TEMPERATURE = 0.1

candidate_pdbs = sorted(glob.glob(f'{MPNN_IN_DIR}/*.pdb'))
print(f'Candidate backbones: {len(candidate_pdbs)}')

# ── ProteinMPNN amino acid order (21 including X) ─────────────────────────────
# From protein_mpnn_utils.py — must match exactly
AA_ORDER = 'ACDEFGHIKLMNPQRSTVWYX'

def make_bias_row(bias_dict):
    """Single residue bias row [21] as Python list for JSON serialization."""
    return [float(bias_dict.get(aa, 0.0)) for aa in AA_ORDER]

def make_bias_matrix(bias_dict, length):
    """[L, 21] as nested Python list — JSON serializable, numpy-loadable."""
    row = make_bias_row(bias_dict)
    return [row for _ in range(length)]

# Amino acid biases
null_bias = {aa: 0.0 for aa in AA_ORDER}
cpp_bias  = {aa: 0.0 for aa in AA_ORDER}
cpp_bias['R'] = 2.0    # Arginine  — membrane penetration
cpp_bias['L'] = 1.5    # Leucine   — hydrophobic face
cpp_bias['K'] = 0.5    # Lysine    — mild charge
cpp_bias['P'] = -1.0   # Proline   — helix breaker
cpp_bias['G'] = -0.5   # Glycine   — too flexible

# ── Step 1: Parse PDB chains ───────────────────────────────────────────────────
parsed_jsonl = f'{LOCAL_DIR}/parsed_chains.jsonl'
print('\nStep 1: Parsing PDB chains...')
result = subprocess.run(
    f'python {HELPER_SCRIPT} '
    f'--input_path {MPNN_IN_DIR} '
    f'--output_path {parsed_jsonl}',
    shell=True, capture_output=True, text=True
)
if result.returncode != 0:
    print(f'FAILED: {result.stderr[-300:]}')
else:
    with open(parsed_jsonl) as f:
        lines = f.readlines()
    first = json.loads(lines[0])
    len_b = len(first.get('seq_chain_B', ''))
    len_c = len(first.get('seq_chain_C', ''))
    print(f'OK — {len(lines)} entries')
    print(f'  Chain A={len(first.get("seq_chain_A",""))} '
          f'B={len_b} C={len_c}')

# ── Step 2: Assign chains ──────────────────────────────────────────────────────
assigned_jsonl = f'{LOCAL_DIR}/assigned_chains.jsonl'
print('\nStep 2: Assigning chains (B+C designed, A fixed)...')
result = subprocess.run(
    f'python {ASSIGN_SCRIPT} '
    f'--input_path {parsed_jsonl} '
    f'--output_path {assigned_jsonl} '
    f'--chain_list "B C"',
    shell=True, capture_output=True, text=True
)
print('OK' if result.returncode == 0 else f'FAILED: {result.stderr[-200:]}')

# ── Step 3: Build bias_by_res as {name: {chain_letter: [[L,21] list]}} ────────
# ProteinMPNN reads this with json.loads then accesses as:
#   bias_by_res_dict[b['name']][letter]  → must be convertible to np.array [L,21]
print('\nStep 3: Building per-residue bias JSONL...')

bias_by_res_path = f'{LOCAL_DIR}/bias_by_res.jsonl'

with open(parsed_jsonl) as infile, \
     open(bias_by_res_path, 'w') as outfile:
    for line in infile:
        entry  = json.loads(line)
        name   = entry['name']
        seq_b  = entry.get('seq_chain_B', '')
        seq_c  = entry.get('seq_chain_C', '')

        # Each chain maps to [L, 21] nested list
        entry_bias = {}
        if seq_b:
            entry_bias['B'] = make_bias_matrix(null_bias, len(seq_b))
        if seq_c:
            entry_bias['C'] = make_bias_matrix(cpp_bias,  len(seq_c))

        # Write one JSON line per entry — ProteinMPNN reads line by line
        outfile.write(json.dumps({name: entry_bias}) + '\n')

# Verify
with open(bias_by_res_path) as f:
    sample = json.loads(f.readline())
sample_name  = list(sample.keys())[0]
sample_chain = list(sample[sample_name].keys())
print(f'OK — sample name: {sample_name}')
print(f'  Chains: {sample_chain}')

if 'B' in sample[sample_name]:
    b_mat = np.array(sample[sample_name]['B'])
    print(f'  Chain B shape: {b_mat.shape}')
    print(f'  Chain B R-bias: {b_mat[0, AA_ORDER.index("R")]}')

if 'C' in sample[sample_name]:
    c_mat = np.array(sample[sample_name]['C'])
    print(f'  Chain C shape: {c_mat.shape}')
    print(f'  Chain C R-bias: {c_mat[0, AA_ORDER.index("R")]}')
    print(f'  Chain C L-bias: {c_mat[0, AA_ORDER.index("L")]}')

# ── Build command ──────────────────────────────────────────────────────────────
cmd_mpnn = (
    f'python {MPNN_SCRIPT} '
    f'--jsonl_path {parsed_jsonl} '
    f'--chain_id_jsonl {assigned_jsonl} '
    f'--bias_by_res_jsonl {bias_by_res_path} '
    f'--out_folder {MPNN_OUT_DIR} '
    f'--num_seq_per_target {NUM_SEQS} '
    f'--sampling_temp {TEMPERATURE} '
    f'--seed 42 '
    f'--batch_size 1 '
    f'--model_name v_48_020 '
    f'--path_to_model_weights /content/ProteinMPNN/vanilla_model_weights '
    f'--save_score 1'
)

print(f'\nCommand built. Run Cell 6.')

Candidate backbones: 100

Step 1: Parsing PDB chains...
OK — 100 entries
  Chain A=50 B=90 C=15

Step 2: Assigning chains (B+C designed, A fixed)...
OK

Step 3: Building per-residue bias JSONL...
OK — sample name: candidate_33_myc_binder_69
  Chains: ['B', 'C']
  Chain B shape: (90, 21)
  Chain B R-bias: 0.0
  Chain C shape: (15, 21)
  Chain C R-bias: 2.0
  Chain C L-bias: 1.5

Command built. Run Cell 6.


In [54]:
# ── Fix: write single JSON not JSONL ──────────────────────────────────────────
import json, numpy as np

AA_ORDER = 'ACDEFGHIKLMNPQRSTVWYX'

def make_bias_matrix(bias_dict, length):
    row = [float(bias_dict.get(aa, 0.0)) for aa in AA_ORDER]
    return [row for _ in range(length)]

null_bias = {aa: 0.0 for aa in AA_ORDER}
cpp_bias  = {aa: 0.0 for aa in AA_ORDER}
cpp_bias['R'] = 2.0
cpp_bias['L'] = 1.5
cpp_bias['K'] = 0.5
cpp_bias['P'] = -1.0
cpp_bias['G'] = -0.5

# Build one flat dict with ALL 100 entries
all_bias = {}
with open(parsed_jsonl) as f:
    for line in f:
        entry = json.loads(line)
        name  = entry['name']
        seq_b = entry.get('seq_chain_B', '')
        seq_c = entry.get('seq_chain_C', '')
        all_bias[name] = {}
        if seq_b:
            all_bias[name]['B'] = make_bias_matrix(null_bias, len(seq_b))
        if seq_c:
            all_bias[name]['C'] = make_bias_matrix(cpp_bias,  len(seq_c))

# Write as single JSON file
bias_by_res_path = f'{LOCAL_DIR}/bias_by_res.jsonl'
with open(bias_by_res_path, 'w') as f:
    json.dump(all_bias, f)

# Verify all 100 entries are present
print(f'Total entries in bias dict: {len(all_bias)}')
sample_name = list(all_bias.keys())[0]
print(f'Sample name: {sample_name}')
print(f'Sample chains: {list(all_bias[sample_name].keys())}')
b_arr = np.array(all_bias[sample_name].get('B', [[0]*21]))
c_arr = np.array(all_bias[sample_name].get('C', [[0]*21]))
print(f'Chain B shape: {b_arr.shape} R-bias: {b_arr[0, AA_ORDER.index("R")]}')
print(f'Chain C shape: {c_arr.shape} R-bias: {c_arr[0, AA_ORDER.index("R")]}')

# Rebuild command
cmd_mpnn = (
    f'python {MPNN_SCRIPT} '
    f'--jsonl_path {parsed_jsonl} '
    f'--chain_id_jsonl {assigned_jsonl} '
    f'--bias_by_res_jsonl {bias_by_res_path} '
    f'--out_folder {MPNN_OUT_DIR} '
    f'--num_seq_per_target {NUM_SEQS} '
    f'--sampling_temp {TEMPERATURE} '
    f'--seed 42 '
    f'--batch_size 1 '
    f'--model_name v_48_020 '
    f'--path_to_model_weights /content/ProteinMPNN/vanilla_model_weights '
    f'--save_score 1'
)
print('\nCommand rebuilt. Run Cell 6.')

Total entries in bias dict: 100
Sample name: candidate_33_myc_binder_69
Sample chains: ['B', 'C']
Chain B shape: (90, 21) R-bias: 0.0
Chain C shape: (15, 21) R-bias: 2.0

Command rebuilt. Run Cell 6.


In [55]:
# ── Cell 6: Run ProteinMPNN sequence design ────────────────────────────────────
import subprocess, time, sys, os, glob

sys.path.insert(0, '/content/ProteinMPNN')
env               = os.environ.copy()
env['PYTHONPATH'] = '/content/ProteinMPNN:' + env.get('PYTHONPATH', '')

print('Starting ProteinMPNN...')
print(f'Designing {len(candidate_pdbs)} backbones × {NUM_SEQS} seqs = '
      f'{len(candidate_pdbs)*NUM_SEQS} total sequences')
print(f'Expected runtime on A100: 15-30 minutes\n')

start     = time.time()
log_lines = []

process = subprocess.Popen(
    cmd_mpnn,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd='/content/ProteinMPNN',
    env=env
)

for line in process.stdout:
    print(line, end='', flush=True)
    log_lines.append(line)

process.wait()
elapsed = (time.time() - start) / 60

os.makedirs(MPNN_OUT_DIR, exist_ok=True)
with open(f'{MPNN_OUT_DIR}/mpnn_run.log', 'w') as f:
    f.writelines(log_lines)

if process.returncode == 0:
    fa_files = glob.glob(f'{MPNN_OUT_DIR}/seqs/*.fa')
    print(f'\nProteinMPNN complete in {elapsed:.1f} min.')
    print(f'FASTA files generated: {len(fa_files)}')
    if len(fa_files) == 0:
        print('\nWARNING: No FASTA files found.')
        print('Output directory structure:')
        for root, dirs, files in os.walk(MPNN_OUT_DIR):
            for fname in files[:20]:
                print(f'  {os.path.join(root, fname)}')
else:
    print(f'\nProteinMPNN FAILED after {elapsed:.1f} min.')
    print('Last 30 lines:')
    print(''.join(log_lines[-30:]))

Starting ProteinMPNN...
Designing 100 backbones × 8 seqs = 800 total sequences
Expected runtime on A100: 15-30 minutes

----------------------------------------
fixed_positions_jsonl is NOT loaded
----------------------------------------
pssm_jsonl is NOT loaded
----------------------------------------
omit_AA_jsonl is NOT loaded
----------------------------------------
bias_AA_jsonl is NOT loaded
----------------------------------------
tied_positions_jsonl is NOT loaded
bias by residue dictionary is loaded
----------------------------------------
discarded {'bad_chars': 0, 'too_long': 0, 'bad_seq_length': 0}
----------------------------------------
Number of edges: 48
Training noise level: 0.2A
Generating sequences for: candidate_33_myc_binder_69
8 sequences of length 155 generated in 5.5163 seconds
Generating sequences for: candidate_85_myc_binder_33
8 sequences of length 133 generated in 4.7116 seconds
Generating sequences for: candidate_23_myc_binder_40
8 sequences of length 156 g

In [56]:
# ── Cell 7: Parse sequences and apply CPP tail composition filter ──────────────
import glob, re

fa_files = sorted(glob.glob(f'{MPNN_OUT_DIR}/seqs/*.fa'))
print(f'FASTA files: {len(fa_files)}')

# CPP tail is always the last 15 residues (chain C)
# Filter: tail must have >= 4 Arg (R) + >= 3 Leu (L) for amphipathic moment
CPP_MIN_ARG = 4
CPP_MIN_LEU = 3

all_sequences = []

for fa_path in fa_files:
    design_id = os.path.basename(fa_path).replace('.fa', '')
    with open(fa_path) as f:
        lines = f.readlines()

    # Parse FASTA — alternating header/sequence lines
    sequences = []
    for i in range(0, len(lines)-1, 2):
        header = lines[i].strip().lstrip('>')
        seq    = lines[i+1].strip() if i+1 < len(lines) else ''
        if seq:
            sequences.append((header, seq))

    for header, seq in sequences:
        # Extract score from header if present
        score_match = re.search(r'score=([\d.]+)', header)
        score       = float(score_match.group(1)) if score_match else 0.0

        # CPP tail = last 15 residues
        cpp_tail    = seq[-15:] if len(seq) >= 15 else seq
        n_arg       = cpp_tail.count('R')
        n_leu       = cpp_tail.count('L')

        # Full sequence stats
        total_charge = seq.count('R') + seq.count('K') - seq.count('D') - seq.count('E')

        all_sequences.append({
            'design_id':   design_id,
            'header':      header,
            'sequence':    seq,
            'length':      len(seq),
            'score':       score,
            'cpp_tail':    cpp_tail,
            'cpp_arg':     n_arg,
            'cpp_leu':     n_leu,
            'net_charge':  total_charge,
            'cpp_pass':    n_arg >= CPP_MIN_ARG and n_leu >= CPP_MIN_LEU,
        })

print(f'Total sequences parsed: {len(all_sequences)}')
print(f'\nCPP tail filter (>= {CPP_MIN_ARG} Arg AND >= {CPP_MIN_LEU} Leu):')
cpp_passed = [s for s in all_sequences if s['cpp_pass']]
cpp_failed = [s for s in all_sequences if not s['cpp_pass']]
print(f'  Passed: {len(cpp_passed)}')
print(f'  Failed: {len(cpp_failed)}')

if len(cpp_passed) == 0:
    print('\nNo sequences passed CPP filter.')
    print('Relaxing to >= 3 Arg AND >= 2 Leu...')
    cpp_passed = [s for s in all_sequences
                  if s['cpp_arg'] >= 3 and s['cpp_leu'] >= 2]
    print(f'  Relaxed filter passed: {len(cpp_passed)}')

print(f'\nCPP tail composition distribution:')
arg_counts = [s['cpp_arg'] for s in all_sequences]
leu_counts = [s['cpp_leu'] for s in all_sequences]
print(f'  Arg in tail: mean={np.mean(arg_counts):.1f} '
      f'range={min(arg_counts)}-{max(arg_counts)}')
print(f'  Leu in tail: mean={np.mean(leu_counts):.1f} '
      f'range={min(leu_counts)}-{max(leu_counts)}')

print(f'\nTop 10 sequences by score:')
top_by_score = sorted(cpp_passed, key=lambda x: x['score'])[:10]
print(f'{"Design":<30} {"Score":>8} {"Arg":>5} {"Leu":>5} {"Charge":>8} {"Length":>8}')
print('-' * 66)
for s in top_by_score:
    print(f'{s["design_id"]:<30} {s["score"]:>8.4f} '
          f'{s["cpp_arg"]:>5} {s["cpp_leu"]:>5} '
          f'{s["net_charge"]:>8} {s["length"]:>8}')

FASTA files: 100
Total sequences parsed: 900

CPP tail filter (>= 4 Arg AND >= 3 Leu):
  Passed: 471
  Failed: 429

CPP tail composition distribution:
  Arg in tail: mean=8.0 range=0-15
  Leu in tail: mean=2.6 range=0-12

Top 10 sequences by score:
Design                            Score   Arg   Leu   Charge   Length
------------------------------------------------------------------
candidate_11_myc_binder_51       0.9510     4     4       -5       96
candidate_11_myc_binder_51       1.0257     4     4       -1       96
candidate_11_myc_binder_51       1.0276     5     3       -3       96
candidate_11_myc_binder_51       1.0339     4     4       -6       96
candidate_11_myc_binder_51       1.0348     6     3        1       96
candidate_78_myc_binder_78       1.0420    10     3        4      110
candidate_78_myc_binder_78       1.0506    10     3        2      110
candidate_11_myc_binder_51       1.0508     4     5       -6       96
candidate_9_myc_binder_96        1.0549     8     3   

In [60]:
# ── Cell 8: ESMFold — stores pLDDT directly ───────────────────────────────────
import torch, os, time, glob

esm_candidates = sorted(cpp_passed, key=lambda x: x['score'])[:50]
print(f'Folding {len(esm_candidates)} sequences...\n')

os.makedirs(ESM_OUT_DIR, exist_ok=True)
start      = time.time()
folded     = 0
failed     = 0
esm_scores = {}   # name → pLDDT stored directly from model output

for i, s in enumerate(esm_candidates):
    seq_name = f'{s["design_id"]}_seq{i}'
    out_path = f'{ESM_OUT_DIR}/{seq_name}.pdb'

    try:
        raw = s['sequence']
        if isinstance(raw, list):
            sequence = ''.join(raw)
        else:
            sequence = raw.strip()
        valid_aa = set('ACDEFGHIKLMNPQRSTVWY')
        sequence = ''.join(c for c in sequence.upper() if c in valid_aa)

        if len(sequence) < 10:
            failed += 1
            continue

        print(f'  {i+1}/{len(esm_candidates)}: {seq_name} len={len(sequence)}...', end=' ', flush=True)

        tokenized = tokenizer(sequence, return_tensors='pt', add_special_tokens=False)
        tokenized = {k: v.cuda() for k, v in tokenized.items()}

        with torch.no_grad():
            output = model(**tokenized)

        pdb_str = model.output_to_pdb(output)[0]

        # Store pLDDT directly from tensor — do not re-read from B-factor
        plddt_per_residue = output.plddt[0].cpu().numpy()   # [L] array 0-1
        mean_plddt        = float(plddt_per_residue.mean())
        esm_scores[seq_name] = {
            'plddt':           mean_plddt,
            'plddt_per_res':   plddt_per_residue.tolist(),
            'length':          len(sequence),
            'design_id':       s['design_id'],
            'sequence':        sequence,
        }

        with open(out_path, 'w') as f:
            f.write(pdb_str)

        print(f'pLDDT {mean_plddt:.3f}')
        folded += 1

    except Exception as e:
        print(f'FAILED: {e}')
        failed += 1

    if i % 10 == 0:
        torch.cuda.empty_cache()

# Save scores to disk so Cell 9 doesn't need to re-read B-factors
import json
with open(f'{ESM_OUT_DIR}/esm_scores.json', 'w') as f:
    scores_serializable = {k: {kk: vv for kk, vv in v.items()
                               if kk != 'plddt_per_res'}
                           for k, v in esm_scores.items()}
    json.dump(scores_serializable, f, indent=2)

elapsed  = (time.time() - start) / 60
esm_pdbs = glob.glob(f'{ESM_OUT_DIR}/*.pdb')
print(f'\nESMFold complete in {elapsed:.1f} min.')
print(f'  Folded:    {folded}')
print(f'  Failed:    {failed}')
print(f'  PDB files: {len(esm_pdbs)}')
print(f'  pLDDT range: {min(v["plddt"] for v in esm_scores.values()):.3f} — '
      f'{max(v["plddt"] for v in esm_scores.values()):.3f}')

Folding 50 sequences...

  1/50: candidate_11_myc_binder_51_seq0 len=95... pLDDT 0.577
  2/50: candidate_11_myc_binder_51_seq1 len=95... pLDDT 0.548
  3/50: candidate_11_myc_binder_51_seq2 len=95... pLDDT 0.621
  4/50: candidate_11_myc_binder_51_seq3 len=95... pLDDT 0.598
  5/50: candidate_11_myc_binder_51_seq4 len=95... pLDDT 0.585
  6/50: candidate_78_myc_binder_78_seq5 len=109... pLDDT 0.773
  7/50: candidate_78_myc_binder_78_seq6 len=109... pLDDT 0.746
  8/50: candidate_11_myc_binder_51_seq7 len=95... pLDDT 0.521
  9/50: candidate_9_myc_binder_96_seq8 len=94... pLDDT 0.788
  10/50: candidate_9_myc_binder_96_seq9 len=94... pLDDT 0.784
  11/50: candidate_51_myc_binder_98_seq10 len=108... pLDDT 0.753
  12/50: candidate_79_myc_binder_39_seq11 len=105... pLDDT 0.740
  13/50: candidate_9_myc_binder_96_seq12 len=94... pLDDT 0.786
  14/50: candidate_11_myc_binder_51_seq13 len=95... pLDDT 0.560
  15/50: candidate_51_myc_binder_98_seq14 len=108... pLDDT 0.757
  16/50: candidate_45_myc_binder

In [62]:
# ── Cell 9: pLDDT-only filter — drop RMSD comparison ─────────────────────────
import glob, json, os
import numpy as np

esm_pdbs = sorted(glob.glob(f'{ESM_OUT_DIR}/*.pdb'))

with open(f'{ESM_OUT_DIR}/esm_scores.json') as f:
    esm_scores = json.load(f)

print(f'ESMFold structures: {len(esm_pdbs)}')

ESM_PLDDT_MIN = 0.70

esm_results = []
for esm_pdb in esm_pdbs:
    seq_name  = os.path.basename(esm_pdb).replace('.pdb', '')
    if seq_name not in esm_scores:
        continue
    entry = {
        'seq_name':  seq_name,
        'esm_pdb':   esm_pdb,
        'esm_plddt': esm_scores[seq_name]['plddt'],
        'design_id': esm_scores[seq_name]['design_id'],
        'length':    esm_scores[seq_name]['length'],
        'sequence':  esm_scores[seq_name].get('sequence', ''),
    }
    entry['pass'] = entry['esm_plddt'] >= ESM_PLDDT_MIN
    esm_results.append(entry)

esm_passed = sorted(
    [r for r in esm_results if r['pass']],
    key=lambda x: -x['esm_plddt']
)
esm_failed = [r for r in esm_results if not r['pass']]

plddt_vals = [r['esm_plddt'] for r in esm_results]

print(f'\nFilter: pLDDT >= {ESM_PLDDT_MIN} (sequence folds into structured protein)')
print(f'  Passed: {len(esm_passed)} / {len(esm_results)}')
print(f'  Failed: {len(esm_failed)}')
print(f'\npLDDT distribution:')
print(f'  Range: {min(plddt_vals):.3f} — {max(plddt_vals):.3f}')
print(f'  Mean:  {np.mean(plddt_vals):.3f}')

print(f'\nAll passing sequences:')
print(f'{"Sequence":<40} {"pLDDT":>8} {"Length":>8}')
print('-' * 58)
for r in esm_passed:
    print(f'{r["seq_name"]:<40} {r["esm_plddt"]:>8.3f} {r["length"]:>8}')

ESMFold structures: 50

Filter: pLDDT >= 0.7 (sequence folds into structured protein)
  Passed: 40 / 50
  Failed: 10

pLDDT distribution:
  Range: 0.521 — 0.818
  Mean:  0.729

All passing sequences:
Sequence                                    pLDDT   Length
----------------------------------------------------------
candidate_1_myc_binder_88_seq34             0.818       82
candidate_45_myc_binder_82_seq37            0.790      109
candidate_9_myc_binder_96_seq8              0.788       94
candidate_1_myc_binder_88_seq24             0.788       82
candidate_9_myc_binder_96_seq12             0.786       94
candidate_9_myc_binder_96_seq9              0.784       94
candidate_9_myc_binder_96_seq30             0.784       94
candidate_81_myc_binder_63_seq41            0.781      109
candidate_78_myc_binder_78_seq18            0.779      109
candidate_9_myc_binder_96_seq23             0.778       94
candidate_36_myc_binder_60_seq36            0.777       68
candidate_75_myc_binder_92_seq27 

In [63]:
# ── Cell 10: Save passing sequences and push to Drive ─────────────────────────
import json, shutil

MPNN_DRIVE_OUT = f'{DRIVE_DIR}/proteinmpnn_outputs'
ESM_DRIVE_OUT  = f'{DRIVE_DIR}/esm_outputs'
OPENMM_IN_DIR  = f'{DRIVE_DIR}/openmm_inputs'

for d in [MPNN_DRIVE_OUT, ESM_DRIVE_OUT, OPENMM_IN_DIR]:
    os.makedirs(d, exist_ok=True)

# Save all passing ESMFold PDBs — these are the inputs to OpenMM
print(f'Saving {len(esm_passed)} ESMFold confirmed structures...')
for r in esm_passed:
    # Save PDB
    shutil.copy2(r['esm_pdb'], f'{OPENMM_IN_DIR}/{r["seq_name"]}.pdb')
    # Save matching FASTA sequence
    seq_entry = next((s for s in all_sequences
                      if r['seq_name'].startswith(s['design_id'])), None)
    if seq_entry:
        with open(f'{OPENMM_IN_DIR}/{r["seq_name"]}.fasta', 'w') as f:
            f.write(f'>{r["seq_name"]}\n{seq_entry["sequence"]}\n')

print(f'  {len(esm_passed)} structures saved to {OPENMM_IN_DIR}')

# Push all ProteinMPNN outputs to Drive
print('\nPushing ProteinMPNN outputs to Drive...')
for fname in os.listdir(MPNN_OUT_DIR):
    src = f'{MPNN_OUT_DIR}/{fname}'
    if os.path.isfile(src):
        shutil.copy2(src, f'{MPNN_DRIVE_OUT}/{fname}')
for fname in os.listdir(ESM_OUT_DIR):
    shutil.copy2(f'{ESM_OUT_DIR}/{fname}', f'{ESM_DRIVE_OUT}/{fname}')

# Update pipeline config
with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

cfg['mpnn_sequences_generated'] = len(all_sequences)
cfg['mpnn_cpp_passed']          = len(cpp_passed)
cfg['esm_evaluated']            = len(esm_results)
cfg['esm_passed']               = len(esm_passed)
cfg['esm_passed_names']         = [r['seq_name'] for r in esm_passed]
cfg['openmm_input_dir']         = OPENMM_IN_DIR

with open(f'{DRIVE_DIR}/pipeline_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'\nPipeline config updated.')
print(f'\nPhase III summary:')
print(f'  RFdiffusion backbones in:        100')
print(f'  ProteinMPNN sequences generated: {len(all_sequences)}')
print(f'  Passed CPP tail filter:          {len(cpp_passed)}')
print(f'  Passed ESMFold confirmation:     {len(esm_passed)}')
print(f'  Forwarded to OpenMM:             {len(esm_passed)}')
print(f'\nOpen OpenMM notebook next.')

Saving 40 ESMFold confirmed structures...
  40 structures saved to /content/drive/MyDrive/myc_gnn/openmm_inputs

Pushing ProteinMPNN outputs to Drive...

Pipeline config updated.

Phase III summary:
  RFdiffusion backbones in:        100
  ProteinMPNN sequences generated: 900
  Passed CPP tail filter:          471
  Passed ESMFold confirmation:     40
  Forwarded to OpenMM:             40

Open OpenMM notebook next.


In [64]:
# ── Save sequence data for OpenMM ─────────────────────────────────────────────
import json, os

OPENMM_IN_DIR = f'{DRIVE_DIR}/openmm_inputs'

# Save full sequence metadata alongside each PDB
seq_metadata = {}
for r in esm_passed:
    seq_metadata[r['seq_name']] = {
        'sequence':  r['sequence'],
        'design_id': r['design_id'],
        'length':    r['length'],
        'esm_plddt': r['esm_plddt'],
    }

with open(f'{OPENMM_IN_DIR}/sequence_metadata.json', 'w') as f:
    json.dump(seq_metadata, f, indent=2)

# Update pipeline config
with open(f'{DRIVE_DIR}/pipeline_config.json') as f:
    cfg = json.load(f)

cfg['openmm_candidates']     = len(esm_passed)
cfg['openmm_top_candidate']  = esm_passed[0]['seq_name']
cfg['openmm_top_plddt']      = esm_passed[0]['esm_plddt']
cfg['openmm_input_dir']      = OPENMM_IN_DIR

with open(f'{DRIVE_DIR}/pipeline_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Sequence metadata saved: {len(seq_metadata)} entries')
print(f'Pipeline config updated.')
print(f'\nTop 5 candidates going into OpenMM:')
for r in esm_passed[:5]:
    print(f'  {r["seq_name"]}: pLDDT {r["esm_plddt"]:.3f} length {r["length"]}')
print(f'\nReady to open OpenMM notebook.')

Sequence metadata saved: 40 entries
Pipeline config updated.

Top 5 candidates going into OpenMM:
  candidate_1_myc_binder_88_seq34: pLDDT 0.818 length 82
  candidate_45_myc_binder_82_seq37: pLDDT 0.790 length 109
  candidate_9_myc_binder_96_seq8: pLDDT 0.788 length 94
  candidate_1_myc_binder_88_seq24: pLDDT 0.788 length 82
  candidate_9_myc_binder_96_seq12: pLDDT 0.786 length 94

Ready to open OpenMM notebook.
